In [ ]:
!pip install -q chromadb
!pip install -q sentence-transformers
!pip install -q transformers
!pip install -q accelerate
!pip install -q pypdf
!pip install -q langchain
!pip install -q langchain-community
!pip install -U langchain langchain-community
!pip show langchain
!pip show langchain-community
!pip show langchain-text-splitters

In [2]:
from google.colab import files

uploaded = files.upload()

Saving Gunavathi Resume.pdf to Gunavathi Resume.pdf
Saving AMRUDAVARSHINI V.pdf to AMRUDAVARSHINI V.pdf
Saving Arul Resume 1.pdf to Arul Resume 1.pdf


In [3]:
from langchain_community.document_loaders import PyPDFLoader

all_docs = []

for pdf_file in uploaded.keys():

    loader = PyPDFLoader(pdf_file)

    docs = loader.load()

    for doc in docs:
        doc.metadata["source"] = pdf_file

    all_docs.extend(docs)

print("Total Pages:", len(all_docs))

/tmp/ipykernel_1986/1064067282.py:1: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyPDFLoader


Total Pages: 5


In [4]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=50
)

chunks = splitter.split_documents(all_docs)

print("Total Chunks:", len(chunks))

Total Chunks: 32


In [5]:
!pip show langchain
!pip show langchain-text-splitters

from langchain_text_splitters import RecursiveCharacterTextSplitter

splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=50
)

chunks = splitter.split_documents(all_docs)

print("Total Chunks:", len(chunks))

Name: langchain
Version: 1.3.7
Summary: Building applications with LLMs through composability
Home-page: https://docs.langchain.com/
Author: 
Author-email: 
License: MIT
Location: /usr/local/lib/python3.12/dist-packages
Requires: langchain-core, langgraph, pydantic
Required-by: 
Name: langchain-text-splitters
Version: 1.1.2
Summary: LangChain text splitting utilities
Home-page: https://docs.langchain.com/
Author: 
Author-email: 
License: MIT
Location: /usr/local/lib/python3.12/dist-packages
Requires: langchain-core
Required-by: langchain-classic
Total Chunks: 32


In [ ]:
from sentence_transformers import SentenceTransformer

embedding_model = SentenceTransformer(
    "sentence-transformers/all-MiniLM-L6-v2"
)

In [7]:
import chromadb

client = chromadb.PersistentClient(
    path="./resume_db"
)

collection = client.get_or_create_collection(
    name="resume_collection"
)

In [8]:
for i, chunk in enumerate(chunks):

    embedding = embedding_model.encode(
        chunk.page_content
    ).tolist()

    collection.add(
        ids=[str(i)],
        documents=[chunk.page_content],
        embeddings=[embedding],
        metadatas=[
            {
                "resume_name":
                chunk.metadata["source"]
            }
        ]
    )

print("Stored Successfully")

Stored Successfully


In [ ]:
from transformers import AutoTokenizer
from transformers import AutoModelForCausalLM

model_name = "Qwen/Qwen2.5-3B-Instruct"

tokenizer = AutoTokenizer.from_pretrained(
    model_name
)

model = AutoModelForCausalLM.from_pretrained(
    model_name,
    device_map="auto"
)

In [10]:
def retrieve_context(query):

    query_embedding = embedding_model.encode(
        query
    ).tolist()

    results = collection.query(
        query_embeddings=[query_embedding],
        n_results=5
    )

    return results

In [11]:
def ask_question(question):

    results = retrieve_context(question)

    context = "\n\n".join(
        results["documents"][0]
    )

    prompt = f"""
You are a resume screening assistant.

Use ONLY the information provided below.

Context:
{context}

Question:
{question}

Answer:
"""

    messages = [
        {
            "role": "user",
            "content": prompt
        }
    ]

    text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )

    inputs = tokenizer(
        text,
        return_tensors="pt"
    ).to(model.device)

    outputs = model.generate(
        **inputs,
        max_new_tokens=300
    )

    answer = tokenizer.decode(
        outputs[0],
        skip_special_tokens=True
    )

    return answer

In [12]:
print("Chunks:", len(chunks))
print("Vectors in DB:", collection.count())

Chunks: 32
Vectors in DB: 32


In [13]:
import torch

print(torch.cuda.is_available())

if torch.cuda.is_available():
    print(torch.cuda.get_device_name(0))

True
Tesla T4


In [14]:
print(
    ask_question(
        "Who has Machine Learning experience?"
    )
)

system
You are Qwen, created by Alibaba Cloud. You are a helpful assistant.
user

You are a resume screening assistant.

Use ONLY the information provided below.

Context:
AREAS OF INTEREST
Artificial Intelligence | Machine Learning & Deep Learning | Data Science & Analytics | Audio Signal Processing | Full Stack
Development | Cloud Computing | UI/UX Design | Natural Language Processing | Healthcare AI
PROFESSIONAL SKILLS
 Strong analytical and problem-solving mindset
 Effective written and verbal communication
 Self-driven with initiative beyond curriculum
 Comfortable building end-to-end systems
independently
 Attention to detail in design and development

4 Full Projects Built in First Year of B.Tech
Independently built and published 4 projects on GitHub spanning ML, cloud, full-stack, and Java
97% Accuracy ML Model — AI Voice Classifier
Achieved near-production-grade accuracy with a self-designed 366-feature pipeline and ensemble model
HSC Science Distinction — 90%
Strong perf

In [15]:
def retrieve_context(query):

    query_embedding = embedding_model.encode(
        query
    ).tolist()

    results = collection.query(
        query_embeddings=[query_embedding],
        n_results=5
    )

    for doc, meta in zip(
        results["documents"][0],
        results["metadatas"][0]
    ):
        print(
            "\nResume:",
            meta["resume_name"]
        )

    return results